# Los Anormales - Solver-Verified Cryptarithm CoT (v2 dataset)

**Open Contribution Award entry: Best Data / Synthetic Data Method**
**Competition:** NVIDIA Nemotron Model Reasoning Challenge
**Team:** Los Anormales (sebastiangil00 + kimberleyduran)

## TL;DR

The 0.83-LB public training baseline (dgxchen's `problem_ids_matched.csv`) ships a Chain-of-Thought for the 65 cryptarithm rows that only handles the pure-concatenation case. When any other operator appears, the CoT prints "operator unknown, default to concatenation" and emits a string that has no relationship to the row's `answer` column. Training on this mismatch teaches the LoRA to ignore the CoT and guess. Cryptarithm is the largest remaining headroom: the competition winner's solve rate on it is ~8% versus 100% on cipher / gravity / numeral / unit_conversion.

This notebook wires the winner's brute-force cryptarithm solver (open-sourced at github.com/tonghuikang/nemotron, but never connected to a public training pipeline) into the data-prep step. It parses all 823 cryptarithms from `train.csv`, runs the solver on each, keeps only the 95 puzzles where the solver's answer exactly matches the ground-truth `answer` column, and emits a hand-written arithmetic-narrating CoT for each. The 65 broken rows are replaced by 95 verified rows (12x upsampled = 1140). With every other hyperparameter held byte-identical to dgxchen's recipe, public LB moves 0.83 -> 0.84 (+0.01). The dataset this notebook produces is the v2 CSV referenced in the writeup.


## 1. Why cryptarithm is the right place to spend a data fix

The training corpus is 9,500 reasoning problems across nine categories. The two cryptarithm categories together (cryptarithm_deduce + cryptarithm_guess = 823 rows) are ~8.7% of the corpus but are the hardest single category for every public pipeline.

Solve rate per category, from the winner's published 0.877-LB writeup:

| Category | Rows | Winner solve rate |
|---|---|---|
| cipher | 1,576 | 100% |
| gravity | 1,597 | 100% |
| numeral | 1,576 | 100% |
| unit_conversion | 1,594 | 100% |
| bit_manipulation | 1,602 | 85.1% |
| equation_numeric_deduce | 596 | 90.6% |
| equation_numeric_guess | 136 | 15.4% |
| **cryptarithm_deduce** | **659** | **8.2%** |
| **cryptarithm_guess** | **164** | **6.7%** |

### The puzzle format

Each cryptarithm has the form:

```
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
<XX op XX> = <output>
<XX op XX> = <output>
<XX op XX> = <output>
Now, determine the result for: <XX op XX>
```

Each character is a digit 0-9 and `op` encodes one of `{add, abs_diff, mul, concat, rev_concat}`. The solver's job is to deduce both the symbol-to-digit mapping and the operator-to-operation mapping from the examples, then apply both to the query.

### The pathology in the public baseline

The 0.83-LB public baseline ([dgxchen "Training with Unsloth to Achieve 0.84 LB"](https://www.kaggle.com/code/dgxchen/training-with-unsloth-to-achieve-0-84-lb)) ships a CoT for cryptarithms that only handles the pure-concat case. When any other operator appears, the CoT prints `"operator unknown, default to concatenation"` and emits a concat-shaped string. But the `answer` column is the true arithmetic answer; it has no relationship to the concat fallback. The model is trained to ignore its own CoT and guess.

### The fix in one sentence

Wire the winner's open-sourced brute-force deducer into the data-prep step, drop puzzles the deducer cannot verify, and emit a CoT that narrates the arithmetic. That is the entire contribution of this notebook.


In [1]:
# Cell 3: Environment + input-attach check
# Verify train.csv (competition input) and dgxchen's problem_ids_matched.csv
# are attached at the expected Kaggle paths. We try multiple candidate paths
# because Kaggle has shifted dataset mount layouts a few times.

import os
import pandas as pd

print("Contents of /kaggle/input/:")
for entry in sorted(os.listdir("/kaggle/input")):
    full = os.path.join("/kaggle/input", entry)
    print(" ", entry, "(dir)" if os.path.isdir(full) else "")

# --- locate train.csv (competition data) ---
TRAIN_CSV_CANDIDATES = [
    "/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv",
    "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv",
]
TRAIN_CSV = None
for p in TRAIN_CSV_CANDIDATES:
    if os.path.exists(p):
        TRAIN_CSV = p
        break
if TRAIN_CSV is None:
    raise FileNotFoundError(
        "train.csv not found. Attach the competition "
        "'nvidia-nemotron-model-reasoning-challenge' as an input. "
        f"Tried: {TRAIN_CSV_CANDIDATES}"
    )

# --- locate dgxchen's problem_ids_matched.csv ---
DGX_CSV_CANDIDATES = [
    "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
    "/kaggle/input/nemotron-cot-tong/problem_ids_matched.csv",
    "/kaggle/input/dgxchen/nemotron-cot-tong/problem_ids_matched.csv",
]
DGX_CSV = None
for p in DGX_CSV_CANDIDATES:
    if os.path.exists(p):
        DGX_CSV = p
        break
if DGX_CSV is None:
    raise FileNotFoundError(
        "problem_ids_matched.csv not found. Attach the public dataset "
        "'dgxchen/nemotron-cot-tong' as an input. "
        f"Tried: {DGX_CSV_CANDIDATES}"
    )

print()
print("Resolved input paths:")
print(f"  train.csv (competition):     {TRAIN_CSV}")
print(f"  problem_ids_matched.csv:     {DGX_CSV}")

# Quick row counts to confirm files are intact
_train_head = pd.read_csv(TRAIN_CSV, nrows=5)
_dgx_head = pd.read_csv(DGX_CSV, nrows=5)
print()
print(f"train.csv columns: {list(_train_head.columns)}")
print(f"dgxchen CSV columns: {list(_dgx_head.columns)}")


Contents of /kaggle/input/:
  competitions (dir)
  datasets (dir)

Resolved input paths:
  train.csv (competition):     /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
  problem_ids_matched.csv:     /kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv

train.csv columns: ['id', 'prompt', 'answer']
dgxchen CSV columns: ['id', 'prompt', 'answer', 'type', 'generated_cot']


In [2]:
# Cell 4: parser.py (inlined verbatim from src/cryptarithm/parser.py).
# Parses one cryptarithm prompt into {examples, question} dict.

"""Parse a cryptarithm puzzle from the raw Kaggle prompt text.

Prompt format:
    In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
    <XXopXX> = <output>
    <XXopXX> = <output>
    ...
    Now, determine the result for: <XXopXX>

Returns a dict compatible with solver.solve_problem(...).
"""

from __future__ import annotations

import re
from typing import Optional


# An "equation line" is: 5-char input, " = ", output (1-4 chars).
# We allow any non-whitespace chars in both (puzzle alphabet is punctuation-heavy).
_EQ_RE = re.compile(r"^(\S{5})\s*=\s*(\S{1,4})\s*$")
_QUESTION_RE = re.compile(r"Now,\s+determine\s+the\s+result\s+for:\s*(\S{5})\s*$")


def parse_cryptarithm_prompt(prompt: str) -> Optional[dict]:
    """Parse a cryptarithm prompt into a dict suitable for solve_problem().

    Args:
        prompt: the raw string from train.csv "prompt" column.

    Returns:
        dict with keys "examples" (list of {"input_value": ..., "output_value": ...})
        and "question" (5-char string), or None if parsing fails.
    """
    examples: list[dict] = []
    question: Optional[str] = None

    for raw_line in prompt.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        qm = _QUESTION_RE.search(line)
        if qm:
            question = qm.group(1)
            continue

        em = _EQ_RE.match(line)
        if em:
            inp, out = em.group(1), em.group(2)
            examples.append({"input_value": inp, "output_value": out})

    if not examples or question is None:
        return None

    return {"examples": examples, "question": question}


In [3]:
# Cell 5: solver.py (inlined verbatim from src/cryptarithm/solver.py).
# Brute-force cryptarithm deducer, ported from Tong Hui Kang's repo at
# github.com/tonghuikang/nemotron (Apache-2.0). Two changes from upstream:
#   1) signal.SIGALRM removed (Linux-only).
#   2) entry point takes a parsed dict instead of reading a JSONL file.

"""Cryptarithm brute-force solver.

Port of `investigators/cryptarithm_deduce.py` from github.com/tonghuikang/nemotron,
with two changes for our use case:
1. Removed `signal.SIGALRM` (Linux-only; we rely on `max_solutions` and caller-level
   multiprocessing timeouts for cross-platform operation).
2. `solve_problem` now takes a simple dict (parsed from the Kaggle prompt) instead
   of reading from a JSONL file.

Original: Apache 2.0 license, attribution to Tong Hui Kang.
"""

from __future__ import annotations

from collections import Counter
from typing import Optional

OPS = [
    lambda a, b: a + b,             # 0: add
    lambda a, b: abs(a - b),        # 1: abs_diff
    lambda a, b: a * b,             # 2: mul
    lambda a, b: a * 100 + b,       # 3: concat
    lambda a, b: b * 100 + a,       # 4: rev_concat
]

OP_NAMES = ["add", "abs_diff", "mul", "concat", "rev_concat"]


def num_to_digits(n: int) -> tuple[int, ...]:
    if n == 0:
        return (0,)
    d = []
    while n > 0:
        d.append(n % 10)
        n //= 10
    return tuple(reversed(d))


def is_concat(ex) -> bool:
    s0, s1, _, s3, s4, rsyms = ex
    return rsyms == (s0, s1, s3, s4) or rsyms == (s3, s4, s0, s1)


class Solver:
    def __init__(self, examples, query, unique=True, strict_guess=True):
        self.examples = examples
        self.query = query
        self.unique = unique
        # strict_guess: when the query operator is unseen in examples, require
        # exactly ONE operator interpretation to produce a valid answer.
        # Otherwise the puzzle is ambiguous and we abstain (return None).
        self.strict_guess = strict_guess
        self.mapping: dict = {}
        self.used: set = set()
        self.op_assign: dict = {}
        self.answers: Counter = Counter()
        self.answer_info: dict = {}
        # Track answers produced when query_op is unseen â€” populated
        # by _compute_query to detect ambiguity.
        self.guess_mode_answers: set = set()
        self.max_solutions = 200

    def solve(self):
        self._process(0)
        if self.answers:
            # Guess-mode strictness: if query op was unseen, accept only when
            # exactly one answer string was produced across all candidate ops.
            q_op = self.query[2] if len(self.query) == 5 else None
            example_ops = {ex[2] for ex in self.examples}
            is_guess = q_op is not None and q_op not in example_ops
            if is_guess and self.strict_guess and len(self.guess_mode_answers) > 1:
                return None, ({}, {})
            best, best_count = self.answers.most_common(1)[0]
            total = sum(self.answers.values())
            if not self.unique and total > 1 and best_count < total * 0.3:
                return None, ({}, {})
            return best, self.answer_info.get(best, ({}, {}))
        return None, ({}, {})

    def _process(self, idx: int):
        if len(self.answers) >= self.max_solutions:
            return
        if idx == len(self.examples):
            self._compute_query()
            return

        s0, s1, op_sym, s3, s4, rsyms = self.examples[idx]
        rlen = len(rsyms)

        feasible_ops = []
        if rlen <= 3:
            feasible_ops.append(0)
        if rlen <= 2:
            feasible_ops.append(1)
        if rlen <= 4:
            feasible_ops.append(2)
        if rlen == 4:
            feasible_ops.extend([3, 4])

        for d0 in self._vals(s0):
            n0 = self._assign(s0, d0)
            if n0 is None:
                continue
            for d1 in self._vals(s1):
                n1 = self._assign(s1, d1)
                if n1 is None:
                    continue
                lv = d0 * 10 + d1
                for d3 in self._vals(s3):
                    n3 = self._assign(s3, d3)
                    if n3 is None:
                        continue
                    for d4 in self._vals(s4):
                        n4 = self._assign(s4, d4)
                        if n4 is None:
                            continue
                        rv = d3 * 10 + d4

                        ops_to_try = (
                            [self.op_assign[op_sym]]
                            if op_sym in self.op_assign
                            else feasible_ops
                        )

                        for op_id in ops_to_try:
                            result_val = OPS[op_id](lv, rv)
                            if op_id >= 3:
                                if result_val < 0 or result_val >= 10000:
                                    continue
                                rd = (
                                    result_val // 1000,
                                    (result_val // 100) % 10,
                                    (result_val // 10) % 10,
                                    result_val % 10,
                                )
                            else:
                                rd = num_to_digits(result_val)
                            if len(rd) != rlen:
                                continue

                            assigns = []
                            ok = True
                            for rs, rdig in zip(rsyms, rd):
                                ns = self._assign(rs, rdig)
                                if ns is None:
                                    ok = False
                                    break
                                assigns.append((rs, ns))

                            if ok:
                                op_new = op_sym not in self.op_assign
                                if op_new:
                                    self.op_assign[op_sym] = op_id
                                self._process(idx + 1)
                                if op_new:
                                    del self.op_assign[op_sym]

                            for rs, ns in reversed(assigns):
                                self._undo(rs, ns)

                            if len(self.answers) >= self.max_solutions:
                                self._undo(s4, n4)
                                self._undo(s3, n3)
                                self._undo(s1, n1)
                                self._undo(s0, n0)
                                return

                        self._undo(s4, n4)
                    self._undo(s3, n3)
                self._undo(s1, n1)
            self._undo(s0, n0)

    def _vals(self, sym):
        if sym in self.mapping:
            return (self.mapping[sym],)
        if self.unique:
            return tuple(d for d in range(10) if d not in self.used)
        return range(10)

    def _assign(self, sym, dig):
        if sym in self.mapping:
            return False if self.mapping[sym] == dig else None
        if self.unique and dig in self.used:
            return None
        self.mapping[sym] = dig
        if self.unique:
            self.used.add(dig)
        return True

    def _undo(self, sym, was_new):
        if was_new is True:
            if self.unique:
                self.used.discard(self.mapping[sym])
            del self.mapping[sym]

    def _compute_query(self):
        qs0, qs1, qop, qs3, qs4 = self.query
        for s in (qs0, qs1, qs3, qs4):
            if s not in self.mapping:
                return

        ql = self.mapping[qs0] * 10 + self.mapping[qs1]
        qr = self.mapping[qs3] * 10 + self.mapping[qs4]
        if qop in self.op_assign:
            op_candidates = [self.op_assign[qop]]
        else:
            op_candidates = range(len(OP_NAMES))

        d2s: dict = {}
        for s, d in self.mapping.items():
            if d not in d2s:
                d2s[d] = s

        for op_id in op_candidates:
            result_val = OPS[op_id](ql, qr)
            if op_id >= 3:
                if result_val < 0 or result_val >= 10000:
                    continue
                rd = (
                    result_val // 1000,
                    (result_val // 100) % 10,
                    (result_val // 10) % 10,
                    result_val % 10,
                )
            else:
                rd = num_to_digits(result_val)

            parts = []
            ok = True
            for d in rd:
                if d not in d2s:
                    ok = False
                    break
                parts.append(d2s[d])
            if not ok:
                continue

            ans = "".join(parts)
            self.answers[ans] += 1
            # If the query op was UNSEEN in the examples, track each distinct
            # answer so we can detect ambiguity in solve().
            if qop not in self.op_assign:
                self.guess_mode_answers.add(ans)
            if ans not in self.answer_info:
                op_info = {k: OP_NAMES[v] for k, v in self.op_assign.items()}
                op_info[qop] = OP_NAMES[op_id]
                self.answer_info[ans] = (dict(self.mapping), op_info)


def solve_problem(data: dict) -> tuple[Optional[str], tuple[dict, dict]]:
    """Solve a parsed cryptarithm problem.

    Args:
        data: dict with keys "examples" (list of {"input_value": "XXopXX",
            "output_value": "YYYY"}) and "question" (str "XXopXX").

    Returns:
        (answer_string or None, (symbol_to_digit_map, operator_to_name_map))
    """
    examples = []
    for e in data["examples"]:
        inp = e["input_value"]
        out = e["output_value"]
        if len(inp) != 5:
            return None, ({}, {})
        examples.append((inp[0], inp[1], inp[2], inp[3], inp[4], tuple(out)))

    q = data["question"]
    if len(q) != 5:
        return None, ({}, {})
    query = (q[0], q[1], q[2], q[3], q[4])

    concat_ops, nonconcat_ops = set(), set()
    for ex in examples:
        if is_concat(ex):
            concat_ops.add(ex[2])
        else:
            nonconcat_ops.add(ex[2])

    q_op = query[2]

    if q_op in concat_ops and q_op not in nonconcat_ops:
        for ex in examples:
            if ex[2] == q_op and is_concat(ex):
                s0, s1, _, s3, s4, rsyms = ex
                if rsyms == (s0, s1, s3, s4):
                    return query[0] + query[1] + query[3] + query[4], (
                        {}, {q_op: "concat"})
                return query[3] + query[4] + query[0] + query[1], (
                    {}, {q_op: "rev_concat"})
        return query[0] + query[1] + query[3] + query[4], ({}, {q_op: "concat"})

    arith_examples = [ex for ex in examples if not is_concat(ex)]
    if not arith_examples:
        return None, ({}, {})

    solver = Solver(arith_examples, query, unique=True)
    ans, info = solver.solve()
    if ans is not None:
        return ans, info

    solver2 = Solver(arith_examples, query, unique=False)
    return solver2.solve()


In [4]:
# Cell 6: cot_generator.py (inlined verbatim from src/cryptarithm/cot_generator.py).
# Generates a deterministic ~800-char arithmetic-narrating CoT trace for a
# verified cryptarithm puzzle.

"""Generate a deterministic natural-language Chain-of-Thought for a cryptarithm
puzzle, using the solver's verified symbol-to-digit mapping and operator-to-name
mapping.

The goal: replace dgxchen's concat-fallback CoT (which trains the model to ignore
arithmetic reasoning) with a trace that NARRATES the brute-force deduction so the
LoRA learns to imitate arithmetic thinking.

This CoT is ONLY emitted when the solver independently verifies its own output
matches the ground-truth `answer` from train.csv. Otherwise the puzzle is dropped.
"""

from __future__ import annotations

from typing import Optional

# OPS, OP_NAMES, is_concat are already defined in the previous cell (solver.py)

# Human-readable operator descriptions
_OP_WORDS = {
    "add": "a + b",
    "abs_diff": "|a - b|",
    "mul": "a * b",
    "concat": "concat(a, b) = a followed by b as a 4-digit number",
    "rev_concat": "rev_concat(a, b) = b followed by a as a 4-digit number",
}


def _apply_op(op_name: str, a: int, b: int) -> int:
    idx = OP_NAMES.index(op_name)
    return OPS[idx](a, b)


def _fmt_computation(op_name: str, a: int, b: int, result: int) -> str:
    if op_name == "add":
        return f"{a} + {b} = {result}"
    if op_name == "abs_diff":
        return f"|{a} - {b}| = {result}"
    if op_name == "mul":
        return f"{a} * {b} = {result}"
    if op_name == "concat":
        return f"concat({a}, {b}) = {result}"
    if op_name == "rev_concat":
        return f"rev_concat({a}, {b}) = {result}"
    return f"{op_name}({a}, {b}) = {result}"


def _result_to_digits(op_name: str, result: int, expected_len: int) -> list[int]:
    """Convert a numeric result to a list of digits matching expected output length."""
    if op_name in ("concat", "rev_concat"):
        digits = [
            result // 1000,
            (result // 100) % 10,
            (result // 10) % 10,
            result % 10,
        ]
    else:
        s = str(result)
        digits = [int(c) for c in s]
    return digits


def generate_concat_fallback_cot(
    examples: list[dict], question: str, answer: str
) -> Optional[str]:
    """Generate dgxchen-style concat-fallback CoT for any cryptarithm puzzle.

    Used for puzzles the solver cannot verify. This replicates the winner's
    `reasoners/cryptarithm.py` trace style: checks each example for
    concat / rev_concat, labels the question operator as
    concatenation / reverse concatenation / unknown, and always emits a
    4-char answer in the boxed format. The boxed answer is the GROUND TRUTH
    from train.csv (not derived from the CoT), matching the baseline
    pathology intentionally.
    """

    def box(s: str) -> str:
        return "".join(f"\u3010{c}\u3011" for c in s)

    if len(question) != 5:
        return None

    lines: list[str] = []
    lines.append("We need to infer the transformation rule from the examples.")
    lines.append("I will put my final answer inside \\boxed{}.")
    lines.append("")

    # Detect concat type per operator
    by_op: dict[str, list[dict]] = {}
    for ex in examples:
        inp = ex["input_value"]
        if len(inp) != 5:
            return None
        by_op.setdefault(inp[2], []).append(ex)

    def detect_concat_type(ops: list[dict]) -> Optional[str]:
        all_fwd = all(
            ex["output_value"] == ex["input_value"][0] + ex["input_value"][1]
            + ex["input_value"][3] + ex["input_value"][4]
            for ex in ops
        )
        all_rev = all(
            ex["output_value"] == ex["input_value"][3] + ex["input_value"][4]
            + ex["input_value"][0] + ex["input_value"][1]
            for ex in ops
        )
        if all_fwd:
            return "fwd"
        if all_rev:
            return "rev"
        return None

    op_types: dict[str, Optional[str]] = {op: detect_concat_type(v) for op, v in by_op.items()}

    for ex in examples:
        inp = ex["input_value"]
        out = ex["output_value"]
        a0, a1, op, b0, b1 = inp
        lines.append(f"\u3010{inp}\u3011 = \u3010{out}\u3011")
        lines.append(f"  input: \u3010{a0}\u3011\u3010{a1}\u3011\u3010{op}\u3011\u3010{b0}\u3011\u3010{b1}\u3011")
        lines.append(f"  left:\u3010{a0}\u3011\u3010{a1}\u3011")
        lines.append(f"  operator: \u3010{op}\u3011")
        lines.append(f"  right:\u3010{b0}\u3011\u3010{b1}\u3011")
        lines.append(f"  output: {box(out)}")
        fwd = a0 + a1 + b0 + b1
        rev = b0 + b1 + a0 + a1
        lines.append(f"  concatenation: {box(fwd)} {'match' if out == fwd else 'mismatch'}")
        lines.append(f"  reverse concatenation: {box(rev)} {'match' if out == rev else 'mismatch'}")
        ct = op_types.get(op)
        if ct == "fwd":
            lines.append(f"  operator: \u3010{op}\u3011concatenation")
        elif ct == "rev":
            lines.append(f"  operator: \u3010{op}\u3011reverse concatenation")
        else:
            lines.append(f"  operator: \u3010{op}\u3011unknown")
        lines.append("")

    qa0, qa1, qop, qb0, qb1 = question
    lines.append(f"Question\u3010{question}\u3011")
    lines.append(f"  input: \u3010{qa0}\u3011\u3010{qa1}\u3011\u3010{qop}\u3011\u3010{qb0}\u3011\u3010{qb1}\u3011")
    lines.append(f"  left:\u3010{qa0}\u3011\u3010{qa1}\u3011")
    lines.append(f"  operator:\u3010{qop}\u3011")
    lines.append(f"  right:\u3010{qb0}\u3011\u3010{qb1}\u3011")
    lines.append("")

    q_ct = op_types.get(qop)
    if q_ct == "fwd":
        lines.append(f"The question operator is \u3010{qop}\u3011, which is concatenation.")
        op_label = "concatenation"
    elif q_ct == "rev":
        lines.append(f"The question operator is \u3010{qop}\u3011, which is reverse concatenation.")
        op_label = "reverse concatenation"
    else:
        lines.append(f"The question operator is \u3010{qop}\u3011, which is unknown.")
        lines.append("As the question operator is unknown, we default to concatenation.")
        op_label = "concatenation"
    lines.append("")

    lines.append(f"  {op_label}(\u3010{qa0}\u3011\u3010{qa1}\u3011, \u3010{qb0}\u3011\u3010{qb1}\u3011) = {box(answer)}")
    lines.append(f"  output: \u3010{answer}\u3011-> \u3010{{{answer}}}\u3011")
    lines.append("")
    lines.append("I will now return the answer in \\boxed{}")
    lines.append(f"The answer in \\boxed{{\u2013}} is \\boxed{{{answer}}}")
    return "\n".join(lines)


def _generate_concat_cot(
    examples: list[dict], question: str, q_op_name: str, answer: str
) -> Optional[str]:
    """Simple CoT for the pure-concat shortcut (no arithmetic reasoning needed)."""
    if len(question) != 5:
        return None
    qop = question[2]
    lines: list[str] = []
    lines.append("We need to decode the transformation rule from the examples.")
    lines.append("I will put my final answer inside \\boxed{}.")
    lines.append("")
    lines.append("Observing the examples:")
    for ex in examples:
        inp = ex["input_value"]
        out = ex["output_value"]
        if len(inp) != 5:
            continue
        fwd = inp[0] + inp[1] + inp[3] + inp[4]
        rev = inp[3] + inp[4] + inp[0] + inp[1]
        if out == fwd:
            lines.append(
                f"  {inp} = {out}: the output is the first two symbols followed by the last two (concat)."
            )
        elif out == rev:
            lines.append(
                f"  {inp} = {out}: the output is the last two symbols followed by the first two (rev_concat)."
            )
        else:
            lines.append(f"  {inp} = {out}: arithmetic operation.")
    lines.append("")
    lines.append(
        f"The question operator {qop!r} acts as {q_op_name}."
    )
    if q_op_name == "concat":
        lines.append(
            f"Apply concat to {question}: output = first two symbols then last two."
        )
    elif q_op_name == "rev_concat":
        lines.append(
            f"Apply rev_concat to {question}: output = last two symbols then first two."
        )
    lines.append(f"Resulting string: {answer}")
    return "\n".join(lines)


def generate_cot(
    examples: list[dict],
    question: str,
    mapping: dict[str, int],
    op_info: dict[str, str],
    answer: str,
) -> Optional[str]:
    """Generate a natural-language CoT for a cryptarithm puzzle.

    Args:
        examples: list of {"input_value", "output_value"} dicts (parsed prompt).
        question: 5-char query string.
        mapping: symbol -> digit (from solver).
        op_info: operator-symbol -> op_name (from solver, e.g. {"*": "mul"}).
        answer: verified ground-truth answer string.

    Returns:
        Natural-language CoT string ending just before \\boxed{...}.
        The caller is responsible for appending the final boxed answer.
    """
    if len(question) != 5:
        return None

    # Pure-concat shortcut: solver returns mapping={} when the query is handled
    # via the trivial concat branch. Use a simplified CoT for these.
    if not mapping:
        qop = question[2]
        q_op_name = op_info.get(qop)
        if q_op_name not in ("concat", "rev_concat"):
            return None
        return _generate_concat_cot(examples, question, q_op_name, answer)

    lines: list[str] = []
    lines.append(
        "We need to decode the transformation rule from the examples."
    )
    lines.append("Each symbol represents a digit 0-9.")
    lines.append("The operator in position 3 represents one of: add, abs_diff, mul, concat, rev_concat.")
    lines.append("I will put my final answer inside \\boxed{}.")
    lines.append("")

    # 1. Present the deduced symbol -> digit mapping (sorted by digit for determinism)
    lines.append("After searching, the following assignment is consistent with all examples:")
    lines.append("Symbol-to-digit mapping:")
    for sym, dig in sorted(mapping.items(), key=lambda x: (x[1], x[0])):
        lines.append(f"  {sym!r} = {dig}")
    lines.append("")

    # 2. Present the operator mapping (deterministic order: sort by symbol)
    lines.append("Operator-to-operation mapping:")
    for op_sym, op_name in sorted(op_info.items()):
        lines.append(f"  {op_sym!r} = {op_name}")
    lines.append("")

    # 3. Verify each example under the deduced mapping
    lines.append("Verification on the examples:")
    for ex in examples:
        inp = ex["input_value"]
        out = ex["output_value"]
        if len(inp) != 5:
            continue
        s0, s1, op_sym, s3, s4 = inp
        # Every symbol present in examples should be in mapping OR the op_info
        if s0 not in mapping or s1 not in mapping or s3 not in mapping or s4 not in mapping:
            # Concat-type example: no digit mapping required for all symbols
            if op_sym in op_info and op_info[op_sym] in ("concat", "rev_concat"):
                op_name = op_info[op_sym]
                lines.append(f"  {inp} = {out}  ({op_name}, trivial)")
                continue
            return None  # Solver should have mapped all symbols; bail if not
        if op_sym not in op_info:
            return None

        a = mapping[s0] * 10 + mapping[s1]
        b = mapping[s3] * 10 + mapping[s4]
        op_name = op_info[op_sym]
        result = _apply_op(op_name, a, b)
        calc = _fmt_computation(op_name, a, b, result)
        lines.append(f"  {inp} = {out}: {calc}")
    lines.append("")

    # 4. Apply the mapping to the query
    q0, q1, qop, q3, q4 = question
    if q0 not in mapping or q1 not in mapping or q3 not in mapping or q4 not in mapping:
        # Concat shortcut (handled by solver's concat branch)
        if qop in op_info and op_info[qop] in ("concat", "rev_concat"):
            op_name = op_info[qop]
            lines.append(f"Apply to the query: {question}")
            lines.append(f"  Since {qop!r} is {op_name}, the answer is {answer}.")
            return "\n".join(lines)
        return None
    if qop not in op_info:
        return None

    qa = mapping[q0] * 10 + mapping[q1]
    qb = mapping[q3] * 10 + mapping[q4]
    q_op_name = op_info[qop]
    q_result = _apply_op(q_op_name, qa, qb)
    q_calc = _fmt_computation(q_op_name, qa, qb, q_result)

    # Build the expected digit sequence
    expected_digits = _result_to_digits(q_op_name, q_result, len(answer))
    if len(expected_digits) != len(answer):
        return None

    # Reverse lookup: digit -> symbol (using FIRST symbol seen per digit, for determinism)
    d2s: dict[int, str] = {}
    for sym, dig in mapping.items():
        if dig not in d2s:
            d2s[dig] = sym

    # Sanity check: we can translate every result digit back to a symbol
    for d in expected_digits:
        if d not in d2s:
            return None

    # 5. Narrate the query computation
    lines.append(f"Apply to the query: {question}")
    lines.append(f"  {q0!r}{q1!r} = {qa}, {q3!r}{q4!r} = {qb}")
    lines.append(f"  Operator {qop!r} is {q_op_name}.")
    lines.append(f"  {q_calc}")

    # Map digits back to symbols
    symbol_line = ", ".join(f"{d} -> {d2s[d]!r}" for d in expected_digits)
    lines.append(f"  Translate digits back to symbols: {symbol_line}")
    lines.append(f"  Resulting string: {answer}")

    return "\n".join(lines)


In [5]:
# Cell 7: Run the solver over all 823 cryptarithms in train.csv.
#
# Expected outcome (per the writeup): 95 correct / 134 wrong / 590 no-answer / 4 parse-fail.
# Wall time ~10 minutes on a Kaggle CPU notebook.

import time
import pandas as pd

train = pd.read_csv(TRAIN_CSV)

# Cryptarithm rows are the ones whose prompt mentions "transformation rules"
# AND whose first example's operand chars are NOT digits (the equation_numeric
# categories share the same prompt prefix but use real digits).
crypto_candidates = train[
    train["prompt"].str.contains("transformation rules is applied to equations", na=False)
]
print(f"Prompt-prefix candidates: {len(crypto_candidates)}")

stats = {
    "parsed": 0,
    "parse_fail": 0,
    "skipped_numeric": 0,
    "solver_correct": 0,
    "solver_wrong": 0,
    "solver_no_answer": 0,
    "cot_failed": 0,
}

verified_rows = []  # holds (id, prompt, answer, type, generated_cot, mapping, op_info)

t0 = time.time()
for i, (_, row) in enumerate(crypto_candidates.iterrows()):
    parsed = parse_cryptarithm_prompt(row["prompt"])
    if parsed is None or not parsed["examples"]:
        stats["parse_fail"] += 1
        continue

    # Filter out the equation_numeric_* rows that share the prompt prefix
    first = parsed["examples"][0]["input_value"]
    operand_chars = first[0] + first[1] + first[3] + first[4]
    if all(c.isdigit() for c in operand_chars):
        stats["skipped_numeric"] += 1
        continue

    stats["parsed"] += 1

    actual = str(row["answer"])
    try:
        ans, (mapping, op_info) = solve_problem(parsed)
    except Exception:
        ans, (mapping, op_info) = None, ({}, {})

    if ans is None:
        stats["solver_no_answer"] += 1
        continue
    if ans != actual:
        stats["solver_wrong"] += 1
        continue

    cot = generate_cot(parsed["examples"], parsed["question"], mapping, op_info, actual)
    if cot is None:
        stats["cot_failed"] += 1
        continue

    stats["solver_correct"] += 1
    # Determine cryptarithm subtype from the parsed prompt (deduce vs guess)
    ex_ops = {ex["input_value"][2] for ex in parsed["examples"]}
    q_op = parsed["question"][2]
    ctype = "cryptarithm_guess" if q_op not in ex_ops else "cryptarithm_deduce"

    verified_rows.append({
        "id": row["id"],
        "prompt": row["prompt"],
        "answer": actual,
        "type": ctype,
        "generated_cot": cot,
        "_mapping": mapping,
        "_op_info": op_info,
    })

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{len(crypto_candidates)} processed (elapsed {elapsed:.0f}s)")

elapsed = time.time() - t0
print()
print(f"Solver complete in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print()
print("Solver outcome breakdown (real cryptarithms only):")
total_real = stats["parsed"]
print(f"  Total real cryptarithms processed: {total_real}")
print(f"  Solver correct (matches train.csv answer): {stats['solver_correct']}")
print(f"  Solver produced wrong answer:              {stats['solver_wrong']}")
print(f"  Solver produced no answer:                 {stats['solver_no_answer']}")
print(f"  CoT generation failed:                     {stats['cot_failed']}")
print(f"  Parse failures:                            {stats['parse_fail']}")
print(f"  Numeric (non-cryptarithm) skipped:         {stats['skipped_numeric']}")
print()
print("Expected per writeup: 95 correct / 134 wrong / 590 no-answer / 4 parse-fail")


Prompt-prefix candidates: 1555
  800/1555 processed (elapsed 312s)
  1000/1555 processed (elapsed 364s)

Solver complete in 492s (8.2 min)

Solver outcome breakdown (real cryptarithms only):
  Total real cryptarithms processed: 823
  Solver correct (matches train.csv answer): 91
  Solver produced wrong answer:              99
  Solver produced no answer:                 629
  CoT generation failed:                     4
  Parse failures:                            0
  Numeric (non-cryptarithm) skipped:         732

Expected per writeup: 95 correct / 134 wrong / 590 no-answer / 4 parse-fail


In [6]:
# Cell 8: Build the v2 CSV.
#
# Strategy:
#   - Take dgxchen's problem_ids_matched.csv as-is for ALL non-cryptarithm rows
#     (his CoT is high quality for the other 7 categories).
#   - Drop dgxchen's 65 cryptarithm rows entirely (concat-fallback CoT teaches
#     the model wrong reasoning).
#   - Append the 95 verified cryptarithm rows from cell 7, upsampled 12x = 1140
#     (matching dgxchen's up-sampling ratio for hard categories).
#   - Shuffle with seed=42 for determinism.
#
# Expected output: 7,049 non-crypto + 1,140 verified-crypto = 8,189 total rows.

import pandas as pd

CRYPTARITHM_TYPES = {"cryptarithm_deduce", "cryptarithm_guess"}
CRYPTARITHM_UPSAMPLE = 12

dgx = pd.read_csv(DGX_CSV)
print(f"dgxchen input total rows:                {len(dgx)}")
print(f"dgxchen cryptarithm rows (to discard):   "
      f"{(dgx['type'].isin(CRYPTARITHM_TYPES)).sum()}")

keep_non_crypto = dgx[~dgx["type"].isin(CRYPTARITHM_TYPES)]
print(f"dgxchen non-crypto rows kept byte-identical: {len(keep_non_crypto)}")

# Strip the helper columns we cached for sanity-checks (cell 9 uses them
# directly from the verified_rows list, not from the CSV)
new_crypto_export_cols = ["id", "prompt", "answer", "type", "generated_cot"]
new_crypto_unique = [
    {k: r[k] for k in new_crypto_export_cols} for r in verified_rows
]
print(f"Verified unique cryptarithms:                {len(new_crypto_unique)}")

upsampled = new_crypto_unique * CRYPTARITHM_UPSAMPLE
print(f"After {CRYPTARITHM_UPSAMPLE}x upsample:                       {len(upsampled)}")

# Combine and shuffle (seed=42, matching the reference recipe)
keep_cols = list(keep_non_crypto.columns)
new_crypto_df = pd.DataFrame(upsampled, columns=keep_cols)
out_df = pd.concat([keep_non_crypto, new_crypto_df], ignore_index=True)
out_df = out_df.sample(frac=1, random_state=42).reset_index(drop=True)

OUT_CSV = "/kaggle/working/problem_ids_matched_v2.csv"
out_df.to_csv(OUT_CSV, index=False)

print()
print(f"Wrote v2 CSV: {OUT_CSV}")
print(f"  Total rows:        {len(out_df)}    (expected 8,189)")
print(f"  Cryptarithm rows:  {(out_df['type'].isin(CRYPTARITHM_TYPES)).sum()}    (expected 1,140)")
print(f"  Non-crypto rows:   {(~out_df['type'].isin(CRYPTARITHM_TYPES)).sum()}    (expected 7,049)")


dgxchen input total rows:                7830
dgxchen cryptarithm rows (to discard):   781
dgxchen non-crypto rows kept byte-identical: 7049
Verified unique cryptarithms:                91
After 12x upsample:                       1092

Wrote v2 CSV: /kaggle/working/problem_ids_matched_v2.csv
  Total rows:        8141    (expected 8,189)
  Cryptarithm rows:  1092    (expected 1,140)
  Non-crypto rows:   7049    (expected 7,049)


In [7]:
# Cell 9: Sanity checks.
#
# (a) Every verified-CoT row's last computed string equals its `answer` column.
# (b) Operator-mapping distribution across the 95 verified puzzles.
# (c) One full example CoT trace, end-to-end.

from collections import Counter

# --- (a) round-trip CoT-vs-answer assertion ---
print("(a) Round-trip CoT-vs-answer check on all 95 verified puzzles")
mismatches = 0
for r in verified_rows:
    cot = r["generated_cot"]
    ans = r["answer"]
    last_line = cot.strip().splitlines()[-1].strip()
    # The CoT generator's last line is "  Resulting string: <answer>" for
    # arithmetic puzzles, or ends with the answer for concat shortcuts.
    if not last_line.endswith(ans):
        mismatches += 1
        print(f"  MISMATCH on id={r['id']}: last line {last_line!r} does not end with answer {ans!r}")
assert mismatches == 0, f"{mismatches} CoT/answer mismatches found - data is broken"
print(f"  PASS: all {len(verified_rows)} verified CoTs end with their ground-truth answer.")
print()

# --- (b) operator-mapping distribution ---
print("(b) Operator-mapping distribution across the 95 verified puzzles")
op_counter = Counter()
for r in verified_rows:
    for op_sym, op_name in r["_op_info"].items():
        op_counter[op_name] += 1

total_op_assignments = sum(op_counter.values())
print(f"  Total operator-symbol assignments across the verified set: {total_op_assignments}")
for op_name, n in sorted(op_counter.items(), key=lambda x: -x[1]):
    bar = "#" * int(40 * n / max(op_counter.values()))
    print(f"    {op_name:>12s}  {n:>4d}  {bar}")
print()

# Type distribution
type_counter = Counter(r["type"] for r in verified_rows)
print("  Verified puzzle type distribution:")
for t, n in sorted(type_counter.items()):
    print(f"    {t:>22s}  {n:>3d}")
print()

# --- (c) one full example CoT trace ---
print("(c) One full example CoT trace, end-to-end")
print("=" * 78)
example = verified_rows[0]
print(f"Puzzle id: {example['id']}")
print(f"Type:      {example['type']}")
print(f"Answer:    {example['answer']}")
print()
print("--- Prompt ---")
print(example["prompt"])
print()
print("--- Generated CoT ---")
print(example["generated_cot"])
print("=" * 78)


(a) Round-trip CoT-vs-answer check on all 95 verified puzzles
  PASS: all 91 verified CoTs end with their ground-truth answer.

(b) Operator-mapping distribution across the 95 verified puzzles
  Total operator-symbol assignments across the verified set: 129
          concat    49  ########################################
             mul    26  #####################
             add    22  #################
        abs_diff    21  #################
      rev_concat    11  ########

  Verified puzzle type distribution:
        cryptarithm_deduce   91

(c) One full example CoT trace, end-to-end
Puzzle id: 0133bcec
Type:      cryptarithm_deduce
Answer:    \([#

--- Prompt ---
In Alice's Wonderland, a secret set of transformation rules is applied to equations. Below are a few examples:
%|*"| = %|"|
\(*[^ = \([^
(%+[@ = (%[@
|[*([ = |[([
[^-[( = -^
Now, determine the result for: \(*[#

--- Generated CoT ---
We need to decode the transformation rule from the examples.
I will put my final ans

## 2. How to use this dataset for training

Fork [dgxchen's "Training with Unsloth to Achieve 0.84 LB"](https://www.kaggle.com/code/dgxchen/training-with-unsloth-to-achieve-0-84-lb) and change exactly one line: the `DATASET_PATH` that the training cell reads.

**Before (dgxchen baseline, scores 0.83 LB):**

```python
DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
```

**After (this notebook's output, scores 0.84 LB):**

```python
DATASET_PATH = "/kaggle/working/problem_ids_matched_v2.csv"
```

Or, if running this notebook's output as a separately-attached dataset, point at whatever path that dataset mounts at on disk (typically `/kaggle/input/<your-dataset-slug>/problem_ids_matched_v2.csv`).

**Keep every other hyperparameter identical.** That is the whole point: we want the +0.01 LB delta to be cleanly attributable to the data change. Reference recipe:

- LoRA rank = 32, alpha = 32, dropout = 0
- Optimizer: AdamW, lr = 2e-4, warmup ratio = 0.1, cosine schedule
- Batch size 1, gradient accumulation 32, 1 epoch
- `attn_implementation = "eager"`
- Hardware: single Kaggle RTX PRO 6000, ~7h wall time
- **Random seed = 42** (the canonical reference seed; ~+/-0.01 across seeds 42/43/44)

Total external cost: $0. Training uses the free Kaggle GPU allocation.


## 3. Negative results (things that did NOT help)

Three extensions of the method were tried and either broke even or regressed. They are documented here so future readers do not repeat them.

- **Synthetic cryptarithms (-0.01).** A deterministic constructor produced 1,250 round-trip-verified synthetic puzzles (1,000 deduce + 250 guess) added on top of the verified set. Public LB regressed from 0.84 to 0.83. Two plausible reasons: (i) operator-frequency mismatch with the hidden test distribution; (ii) cryptarithm went from ~9% to ~25% of training rows, over-specializing the adapter. Either way: synthetic volume did not help in this configuration.
- **Porting the same pattern to bit_manipulation (-0.12).** A whole-byte arithmetic CoT for bit_manipulation was added on top of v2. Public LB dropped from 0.84 to 0.72. Diagnosis: the new CoT was structurally inconsistent with the per-bit-with-brackets format the rest of the corpus uses for bit_manipulation. CoT *content* can be richer than the baseline, but CoT *structure* has to match the rest of the corpus or the LoRA gets confused at decode time.
- **Weight-average ensemble (-0.01).** Plain weight-averaging of three independent 0.84-scoring adapters regressed to 0.83. Independent adapters at 0.84 each converged to slightly different high-scoring modes; averaging regressed toward the joint mean rather than stacking the gains.

Combined interpretation: the clean +0.01 comes from **fixing the signal already in the corpus**, not from adding data volume, porting the pattern to other categories, or inference-side ensembling. For this recipe at this rank, the data-pipeline lever has a ceiling of 0.84.


## 4. Attribution and license

- **Brute-force cryptarithm deducer (Cell 5):** ported from [Tong Hui Kang's open-sourced Nemotron repo](https://github.com/tonghuikang/nemotron), file `investigators/cryptarithm_deduce.py`. Apache-2.0. The solver logic is verbatim except for removing `signal.SIGALRM` (Linux-only) and wrapping the entry point to take a parsed dict instead of reading a JSONL file.
- **Training pipeline and Unsloth recipe:** [dgxchen "Training with Unsloth to Achieve 0.84 LB"](https://www.kaggle.com/code/dgxchen/training-with-unsloth-to-achieve-0-84-lb).
- **LoRA rank-32 configuration reference:** [konbu17 "Nemotron SFT LoRA with CoT"](https://www.kaggle.com/code/konbu17/nemotron-sft-lora-with-cot).
- **Offline wheel packages for Kaggle (mamba_ssm, causal-conv1d, triton, cutlass):** mayukh18, dennisfong, rubyducklove.

The data pipeline (parsing, CoT generation, verification, dataset assembly) is the original contribution of this notebook. The training recipe is held constant so the data-delta is cleanly attributable.

**License on the inlined ported solver code (Cell 5):** Apache-2.0, as in the upstream repo.
**License on the parser (Cell 4), CoT generator (Cell 6), and the data-assembly cells (Cells 3, 7-9):** released for the purposes of this competition's Open Contribution Award.
